# MIST-HER2: Phikon-guided conditional flow

Noise-to-IHC generation with frozen Phikon features, a local H&E pyramid and auxiliary structure/stain losses. Saved metrics cover 12 validation crops.


In [ ]:
SESSION_STEPS = 6_000
RESUME_FROM = None          # /kaggle/input/.../latest.pt; 0 steps = inference only
EXPERIMENT = "full"         # fm-only / structure / full / cnn-only / full-affine
MICRO_BATCH = 4             # эффективный batch всегда 4; больше — если хватает VRAM
PHIKON_PATH = "/kaggle/input/datasets/vasiliygorelov/rrrrrrrrrr"          # offline папка HF Phikon; None = поиск в Input / download
SMOKE_TEST = False

## Experiment modes

`fm-only`: FM loss only. `structure`: add H&E content and structure. `full`: add semantic matching and DAB. `cnn-only`: remove Phikon conditioning, retaining feature losses. `full-affine`: add train-only affine registration. These are available modes, not evidence that every mode was run.


In [ ]:
import importlib.metadata
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

INPUT_ROOT = Path(os.environ.get("IHC_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("IHC_WORK_ROOT", "/kaggle/working"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(WORK_ROOT / ".matplotlib"))
# POT's exact EMD uses CPU; don't initialize unused ML backends on the GPU.
for backend in ("JAX", "TENSORFLOW", "CUPY"):
    os.environ[f"POT_BACKEND_DISABLE_{backend}"] = "1"

markers = list(INPUT_ROOT.rglob("kaggle_bundle.json"))
if len(markers) != 1:
    raise FileNotFoundError("Подключите один распакованный MIST dataset с kaggle_bundle.json")
BUNDLE = markers[0].parent
sys.path.insert(0, str(BUNDLE / "code"))
DATA_ROOT = BUNDLE / "data" / "HER2"
MANIFEST = BUNDLE / "manifests" / "group-disjoint" / "pairs.jsonl"

required_versions = {"torchcfm": "1.0.7", "POT": "0.9.6.post1"}
def installed(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if any(installed(name) != version for name, version in required_versions.items()):
    wheel_markers = list(INPUT_ROOT.rglob("torchcfm_wheels.json"))
    if not wheel_markers:
        archives = list(INPUT_ROOT.rglob("torchcfm-kaggle-wheels.zip"))
        if len(archives) == 1:
            extracted = Path(tempfile.gettempdir()) / "ihc-otcfm-wheels"
            extracted.mkdir(exist_ok=True)
            with zipfile.ZipFile(archives[0]) as archive:
                for info in archive.infolist():
                    if not (extracted / info.filename).resolve().is_relative_to(extracted.resolve()):
                        raise ValueError("Unsafe dependency ZIP path")
                archive.extractall(extracted)
            wheel_markers = list(extracted.rglob("torchcfm_wheels.json"))
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
               "torchcfm==1.0.7", "POT==0.9.6.post1"]
    if wheel_markers:
        command += ["--no-index", "--find-links", str(wheel_markers[0].parent)]
    subprocess.run(command, check=True)

if not SMOKE_TEST and installed("transformers") != "4.57.6":
    subprocess.run([sys.executable,"-m","pip","install","--quiet","transformers==4.57.6"],check=True)
if installed("opencv-python-headless") is None and importlib.util.find_spec("cv2") is None:
    subprocess.run([sys.executable,"-m","pip","install","--quiet","opencv-python-headless==4.12.0.88"],check=True)
print("torchCFM:",installed("torchcfm"),"Transformers:",installed("transformers"))

In [ ]:
"""V4: independent conditional FM, spatial pathology features and weak-pair endpoint losses."""

from __future__ import annotations

import copy
import json
import math
import random
from itertools import pairwise
from pathlib import Path

import numpy as np
import torch
from ihc_flow.data import PairedDataset, TrainingBatches
from ihc_flow.download import sha256_file
from ihc_flow.model import BlockStack, TimeEmbedding, group_norm
from ihc_flow.runtime import initial_noise, save_image, seed_everything, write_json
from skimage.color import hed_from_rgb
from skimage.metrics import structural_similarity
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from torch.utils.data import DataLoader
from torchcfm.conditional_flow_matching import (
    ConditionalFlowMatcher,
)
from tqdm.auto import tqdm


def recipe(smoke=False, experiment="full"):
    if experiment not in {"fm-only", "structure", "full", "cnn-only", "full-affine"}:
        raise ValueError("Unknown experiment")
    return {
        "method": "structure_preserving_cfm_v4",
        "source": "independent_gaussian",
        "experiment": experiment,
        "sigma": 0.0,
        "torchcfm": "1.0.7",
        "backbone": "smoke_stub" if smoke else "owkin/phikon",
        "crop": 64 if smoke else 512,
        "widths": (8, 16) if smoke else (32, 64, 128, 256),
        "batch": 4,
        "total_steps": 8 if smoke else 20000,
        "lr": 0.001 if smoke else 0.0001,
        "warmup": 0 if smoke else 500,
        "ema": 0.9 if smoke else 0.995,
        "validate_every": 4 if smoke else 500,
        "validation_count": 4 if smoke else 12,
        "ode_steps": 4 if smoke else 32,
        "rollout_steps": 2 if smoke else 4,
        "aux_ramp": 1 if smoke else 500,
        "limit_train": 32 if smoke else None,
        "seed": 42,
        "loss_weights": {"content": 1.0, "structure": 0.5, "semantic": 0.5, "dab": 1.0},
    }


def rgb(image):
    return (image.detach().float().cpu().clamp(-1, 1).permute(1, 2, 0).numpy() + 1) / 2

## Frozen pathology features

Phikon supplies 14×14 patch tokens from a 224-pixel input. Backbone weights stay frozen; gradients through generated images still reach the generator. Features on IHC are an exploratory proxy. [Model and license](https://huggingface.co/owkin/phikon).


In [ ]:
class FrozenPhikon(nn.Module):
    """Frozen weights, but differentiable w.r.t. generated RGB for content losses.

    H&E uses no_grad at call sites. Never put generated-image features in no_grad.
    Phikon is pretrained on H&E; cross-stain features here are an experimental proxy.
    """

    channels = 768

    def __init__(self, directory):
        super().__init__()
        from transformers import AutoImageProcessor, ViTModel

        directory = Path(directory)
        processor = AutoImageProcessor.from_pretrained(directory, local_files_only=True)
        self.encoder = (
            ViTModel.from_pretrained(
                directory, add_pooling_layer=False, local_files_only=True
            )
            .eval()
            .requires_grad_(False)
        )
        self.register_buffer(
            "mean", torch.tensor(processor.image_mean).view(1, 3, 1, 1)
        )
        self.register_buffer("std", torch.tensor(processor.image_std).view(1, 3, 1, 1))
        weights = sorted(directory.glob("*.safetensors")) or sorted(
            directory.glob("pytorch_model*.bin")
        )
        if not weights:
            raise FileNotFoundError("No Phikon weight file")
        import hashlib

        digest = hashlib.sha256()
        for path in [
            directory / "config.json",
            directory / "preprocessor_config.json",
            *weights,
        ]:
            digest.update(bytes.fromhex(sha256_file(path)))
        self.identity = digest.hexdigest()

    def train(self, mode=True):
        return super().train(False)

    def forward(self, image):
        # Bound RGB only inside feature extraction. ODE state remains unclipped.
        pixels = F.interpolate(
            (image.float().clamp(-1, 1) + 1) / 2,
            (224, 224),
            mode="bicubic",
            align_corners=False,
            antialias=True,
        )
        tokens = self.encoder(
            pixel_values=(pixels - self.mean) / self.std
        ).last_hidden_state[:, 1:]
        return tokens.transpose(1, 2).reshape(len(image), self.channels, 14, 14)


class SmokeBackbone(nn.Module):
    """Deterministic tiny test double. It is NOT a trained pathology encoder."""

    channels = 8
    identity = "smoke-backbone-v4-fixed"

    def __init__(self):
        super().__init__()
        with torch.random.fork_rng():
            torch.manual_seed(1729)
            self.projection = nn.Conv2d(3, self.channels, 3, padding=1)
        self.eval().requires_grad_(False)

    def forward(self, image):
        return self.projection(F.adaptive_avg_pool2d(image, (8, 8)))


def download_phikon(directory):
    """Explicit public model download; supports a user-provided offline HF directory."""
    from huggingface_hub import snapshot_download

    snapshot_download(
        "owkin/phikon",
        revision="057cc0295895c2df3dd7681a89680da6015cbefe",
        local_dir=str(directory),
        allow_patterns=["config.json", "preprocessor_config.json", "model.safetensors"],
    )
    return Path(directory)


def load_backbone(device, smoke=False, directory=None):
    if smoke:
        return SmokeBackbone().to(device)
    if directory is None:
        raise ValueError("Provide the downloaded/offline Phikon directory")
    return FrozenPhikon(directory).to(device)


class SpatialModulation(nn.Module):
    """Learn spatial gamma/beta from dense frozen pathology tokens at each scale."""

    def __init__(self, width, token_dim):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Conv2d(token_dim, 32, 1),
            nn.SiLU(),
            nn.Conv2d(32, 2 * width, 3, padding=1),
        )
        self.norm = group_norm(width)

    def forward(self, feature, tokens):
        modulation = F.interpolate(
            self.projection(tokens),
            size=feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        gamma, beta = modulation.chunk(2, dim=1)
        return feature + self.norm(feature) * (0.5 * gamma.tanh()) + beta

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and not SMOKE_TEST:
    raise RuntimeError("Включите GPU, либо SMOKE_TEST=True для технической проверки")
torch.set_num_threads(4)
SETTINGS = recipe(SMOKE_TEST, EXPERIMENT)
RUN = WORK_ROOT / f"her2-flow-v4-{EXPERIMENT}{'-smoke' if SMOKE_TEST else ''}"
if SMOKE_TEST:
    BACKBONE = load_backbone(DEVICE,smoke=True)
else:
    directory = Path(PHIKON_PATH) if PHIKON_PATH else None
    if directory is None:
        candidates = [p.parent for p in INPUT_ROOT.rglob("config.json")
                      if "phikon" in p.as_posix().lower() and (p.parent/"model.safetensors").exists()]
        if len(candidates)>1:raise ValueError("Несколько Phikon: задайте PHIKON_PATH")
        directory = candidates[0] if candidates else WORK_ROOT/"phikon"
        if not (directory/"model.safetensors").exists():
            archives=list(INPUT_ROOT.rglob("phikon-kaggle.zip"))
            if len(archives)==1:
                with zipfile.ZipFile(archives[0]) as archive:
                    for info in archive.infolist():
                        if not (directory/info.filename).resolve().is_relative_to(directory.resolve()):
                            raise ValueError("Unsafe model ZIP")
                    archive.extractall(directory)
            else:download_phikon(directory)
    BACKBONE = load_backbone(DEVICE,directory=directory)
    print("Phikon:",directory,"SHA256:",BACKBONE.identity)
print("Device:",DEVICE,"Run:",RUN)
audit=json.loads((MANIFEST.parent/"audit.json").read_text())
assert not any(audit["group_overlap"].values())
print("Group-disjoint counts:",audit["counts"])

## Endpoint losses

Content and structure losses compare generated tissue with H&E. Local semantic matching and DAB statistics compare it with IHC while allowing limited spatial mismatch. They do not correct different fields of view or establish clinical HER2 scores.


In [ ]:
def stain_channels(image):
    """Differentiable standard HED color deconvolution, matched to skimage rgb2hed.

    Fixed stain vectors and thresholds are research proxies, not clinical labels.
    A 1e-3 RGB floor bounds logarithm gradients; range penalty handles saturation.
    """
    raw = (image.float() + 1) / 2
    bounded = raw.clamp(1e-3, 1)
    optical_density = bounded.log() / math.log(1e-6)
    matrix = torch.as_tensor(hed_from_rgb, device=image.device, dtype=torch.float32)
    return torch.einsum("bchw,cd->bdhw", optical_density, matrix).clamp_min(0)


def local_nuclei(image):
    h = stain_channels(image)[:, 0:1]
    # Local contrast reduces stain-strength differences between H&E and IHC.
    h = F.adaptive_avg_pool2d(h, (min(64, h.shape[-2]), min(64, h.shape[-1])))
    background = F.avg_pool2d(h, 9, stride=1, padding=4, count_include_pad=False)
    contrast = h - background
    scale = contrast.square().mean((2, 3), keepdim=True).sqrt().clamp_min(0.005)
    return torch.tanh(contrast / scale)


def tissue_probability(image):
    # Color-only background proxy; computed independently for each image.
    return torch.sigmoid(((1 - image.float().mean(1, keepdim=True)) / 2 - 0.08) / 0.03)


def dab_statistics(image):
    dab = stain_channels(image)[:, 2:3]
    mask = tissue_probability(image)
    denominator = mask.sum((2, 3)).clamp_min(1)
    mean = (dab * mask).sum((2, 3)) / denominator
    positive = (torch.sigmoid((dab - 0.05) / 0.01) * mask).sum((2, 3)) / denominator
    # Soft bins keep the full intensity constraint differentiable.
    bins = torch.linspace(0, 0.3, 16, device=image.device).reshape(1, 16, 1, 1)
    small = F.adaptive_avg_pool2d(dab, (64, 64))
    weights = torch.softmax(-(small - bins).square() / (2 * 0.02**2), dim=1)
    support = F.adaptive_avg_pool2d(mask, (64, 64))
    histogram = (weights * support).sum((2, 3)) / support.sum((2, 3)).clamp_min(1)
    return mean, positive, histogram


def patch_content_loss(generated, condition):
    """Same-location dense feature matching: geometry follows H&E, not the IHC target."""
    generated = F.normalize(generated.float(), dim=1)
    condition = F.normalize(condition.float().detach(), dim=1)
    return (1 - (generated * condition).sum(1)).mean()


def semantic_region_loss(generated, reference, radius=2):
    """Bidirectional local feature correspondence, without pixel-aligned RGB targets.

    Smooth confidence suppresses weak matches; radius permits moderate shifts.
    Does not solve unrelated fields of view or missing cells.
    """
    a, b = (F.normalize(t.float(), dim=1) for t in (generated, reference.detach()))

    def directed(query, bank):
        batch, channels, height, width = query.shape
        candidates = F.unfold(bank, 2 * radius + 1, padding=radius).reshape(
            batch, channels, (2 * radius + 1) ** 2, height * width
        )
        similarities = (query.flatten(2).unsqueeze(2) * candidates).sum(1)
        valid = (
            F.unfold(torch.ones_like(bank[:, :1]), 2 * radius + 1, padding=radius) > 0
        )
        similarities = similarities.masked_fill(~valid, -2)
        best = similarities.max(1).values
        confidence = torch.sigmoid((best.detach() - 0.4) / 0.1)
        return ((1 - best) * confidence).mean()

    return (directed(a, b) + directed(b, a)) / 2


def endpoint_losses(prediction, he, reference, backbone, he_tokens, semantic=True):
    """All terms act on an actual noise-only rollout, not teacher x_t containing IHC."""
    generated_tokens = backbone(prediction)
    structure = F.smooth_l1_loss(local_nuclei(prediction), local_nuclei(he).detach())
    structure = structure + F.mse_loss(
        F.adaptive_avg_pool2d(tissue_probability(prediction), (32, 32)),
        F.adaptive_avg_pool2d(tissue_probability(he), (32, 32)).detach(),
    )
    # Penalize out-of-range RGB separately; do not hide exploding endpoints by clipping.
    structure = structure + 0.1 * F.relu(prediction.abs() - 1).square().mean()
    losses = {
        "content": patch_content_loss(generated_tokens, he_tokens),
        "structure": structure,
    }
    if semantic:
        with torch.no_grad():
            reference_tokens = backbone(reference)
            target_stats = dab_statistics(reference)
        losses["semantic"] = semantic_region_loss(generated_tokens, reference_tokens)
        losses["dab"] = sum(
            F.smooth_l1_loss(a, b)
            for a, b in zip(dab_statistics(prediction), target_stats, strict=True)
        )
    return losses


def proxy_scores(prediction, he, reference):
    with torch.no_grad():
        pred, condition, target = (x.unsqueeze(0) for x in (prediction, he, reference))
        mean, positive, _ = dab_statistics(pred)
        real_mean, real_positive, _ = dab_statistics(target)
        return {
            "dab_mean": mean.item(),
            "real_dab_mean": real_mean.item(),
            "dab_mean_error": (mean - real_mean).abs().item(),
            "dab_positive_fraction": positive.item(),
            "real_dab_positive_fraction": real_positive.item(),
            "he_structure_error": F.smooth_l1_loss(
                local_nuclei(pred), local_nuclei(condition)
            ).item(),
        }

## Model and sampling

The U-Net uses H&E, local features and Phikon modulation. Training starts from independent Gaussian noise. Auxiliary losses use an Euler4 rollout; inference defaults to Heun32. This sampling difference needs validation.


In [ ]:
class HEEncoder(nn.Module):
    """A spatial feature pyramid that never receives x_t or time."""

    def __init__(self, widths):
        super().__init__()
        self.input = nn.Conv2d(3, widths[0], 3, padding=1)
        self.blocks = nn.ModuleList(
            [
                nn.Sequential(
                    group_norm(c),
                    nn.SiLU(),
                    nn.Conv2d(c, c, 3, padding=1),
                    group_norm(c),
                    nn.SiLU(),
                    nn.Conv2d(c, c, 3, padding=1),
                )
                for c in widths
            ]
        )
        self.down = nn.ModuleList(
            [nn.Conv2d(a, b, 3, stride=2, padding=1) for a, b in pairwise(widths)]
        )

    def forward(self, he):
        hidden, features = self.input(he), []
        for level, block in enumerate(self.blocks):
            hidden = hidden + block(hidden)
            features.append(hidden)
            if level < len(self.down):
                hidden = self.down[level](hidden)
        return features


class UNet(nn.Module):
    """Predict RGB velocity from x_t, t and multiscale H&E features."""

    def __init__(self, widths=(32, 64, 128, 256), token_dim=768, use_foundation=True):
        super().__init__()
        time_dim = widths[0] * 4
        self.he_encoder = HEEncoder(widths)
        self.use_foundation = use_foundation
        self.spatial = nn.ModuleList([SpatialModulation(c, token_dim) for c in widths])
        self.time = TimeEmbedding(time_dim)
        self.input = nn.Conv2d(6, widths[0], 3, padding=1)
        self.fusion = nn.ModuleList([nn.Conv2d(2 * c, c, 1) for c in widths])
        self.encoder = nn.ModuleList([BlockStack(c, c, 2, time_dim, 0) for c in widths])
        self.down = nn.ModuleList(
            [nn.Conv2d(a, b, 3, stride=2, padding=1) for a, b in pairwise(widths)]
        )
        self.middle = BlockStack(widths[-1], widths[-1], 2, time_dim, 0)
        reverse = tuple(reversed(widths))
        self.decoder = nn.ModuleList(
            [
                BlockStack((reverse[i - 1] if i else c) + 2 * c, c, 2, time_dim, 0)
                for i, c in enumerate(reverse)
            ]
        )
        self.output = nn.Sequential(
            group_norm(widths[0]), nn.SiLU(), nn.Conv2d(widths[0], 3, 3, padding=1)
        )
        nn.init.zeros_(self.output[-1].weight)
        nn.init.zeros_(self.output[-1].bias)

    def condition(self, he, tokens):
        features = self.he_encoder(he)
        if self.use_foundation:
            features = [
                block(feature, tokens)
                for block, feature in zip(self.spatial, features, strict=True)
            ]
        return features

    def forward(self, image, time, he, features=None):
        if image.shape != he.shape or image.ndim != 4 or image.shape[1] != 3:
            raise ValueError("x_t and H&E must have matching [B,3,H,W] shapes")
        if features is None:
            raise ValueError(
                "Pass condition features produced with this H&E and its tokens"
            )
        embedding = self.time(time.reshape(-1).expand(len(image)))
        hidden, skips = self.input(torch.cat((image, he), 1)), []
        for level, (fusion, block) in enumerate(
            zip(self.fusion, self.encoder, strict=True)
        ):
            hidden = block(fusion(torch.cat((hidden, features[level]), 1)), embedding)
            skips.append(hidden)
            if level < len(self.down):
                hidden = self.down[level](hidden)
        hidden = self.middle(hidden, embedding)
        for block, skip, feature in zip(
            self.decoder, reversed(skips), reversed(features), strict=True
        ):
            hidden = F.interpolate(hidden, size=skip.shape[-2:], mode="nearest")
            hidden = block(torch.cat((hidden, skip, feature), 1), embedding)
        return self.output(hidden)


def flow_batch(matcher, he, ihc):
    """Independent Gaussian; target and its H&E retain their original association."""
    source = torch.randn_like(ihc)
    time, state, velocity = matcher.sample_location_and_conditional_flow(source, ihc)
    return time, state, velocity, he


def rollout(model, he, noise, features, steps=4, checkpointed=True):
    """Differentiable Euler endpoint, with no real IHC anywhere in its inputs."""
    image = noise
    for step in range(steps):
        time = torch.full((len(image),), step / steps, device=image.device)
        if checkpointed and torch.is_grad_enabled():
            # Explicit time argument avoids closure binding all recomputations to the last time.
            velocity = activation_checkpoint(
                model, image, time, he, features, use_reentrant=False
            )
        else:
            velocity = model(image, time, he, features)
        image = image + velocity / steps
    return image


@torch.inference_mode()
def generate(model, backbone, he, noise, steps=32, trajectory=False, solver="heun"):
    if steps < 1 or model.training or solver not in {"euler", "heun"}:
        raise ValueError("Use eval(), positive steps and euler/heun")
    if he.shape != noise.shape:
        raise ValueError("Noise and H&E shapes differ")
    image, delta = noise.clone(), 1 / steps
    features = model.condition(he, backbone(he))
    frames = [(0.0, image.cpu().clone())] if trajectory else []
    frame_steps = {max(1, round(steps * f)) for f in (0.25, 0.5, 0.75, 1)}
    for step in range(steps):
        time = torch.full((len(image),), step / steps, device=image.device)
        first = model(image, time, he, features)
        if solver == "heun":
            second = model(image + delta * first, time + delta, he, features)
            image = image + delta * (first + second) / 2
        else:
            image = image + delta * first
        if trajectory and step + 1 in frame_steps:
            frames.append(((step + 1) / steps, image.cpu().clone()))
    if not torch.isfinite(image).all():
        raise FloatingPointError("Non-finite ODE endpoint")
    return (image, frames) if trajectory else image

## Optional train-only registration

`full-affine` uses conservative ECC alignment before cropping. Validation images stay unchanged. Affine registration cannot recover cells absent from one section.


In [ ]:
"""Conservative training-only affine registration on hematoxylin proxies.

Rejected transforms return the original pair. No registration runs at inference.
This cannot align unrelated fields of view or recreate absent cells.
"""

from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import torch
from ihc_flow.data import PairedDataset
from PIL import Image
from skimage.color import hed_from_rgb


def align_pair(he, ihc):
    import cv2

    if he.shape != ihc.shape or he.ndim != 3:
        raise ValueError("Equal RGB image shapes required")
    height, width = he.shape[:2]
    if height != width:
        raise ValueError("This affine recipe expects square MIST patches")

    def proxy(image):
        optical = np.log(np.maximum(image.astype(np.float64) / 255, 1e-6)) / np.log(
            1e-6
        )
        channel = np.maximum((optical * hed_from_rgb[:, 0]).sum(-1), 0)
        channel = cv2.resize(channel, (256, 256), interpolation=cv2.INTER_AREA)
        channel = cv2.GaussianBlur(channel, (5, 5), 0)
        channel = (channel - channel.mean()) / max(channel.std(), 1e-4)
        return channel.astype(np.float32)

    template, moving = proxy(he), proxy(ihc)
    before = float(np.corrcoef(template.ravel(), moving.ravel())[0, 1])
    report = {
        "accepted": False,
        "before_correlation": before if np.isfinite(before) else None,
    }
    try:
        correlation, warp = cv2.findTransformECC(
            template,
            moving,
            np.eye(2, 3, dtype=np.float32),
            cv2.MOTION_AFFINE,
            (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 1e-5),
            None,
            5,
        )
    except cv2.error:
        return ihc, report
    singular = np.linalg.svd(warp[:, :2], compute_uv=False)
    # Conservative geometry: reject flips, large zoom/shear and translations.
    safe = (
        np.linalg.det(warp[:, :2]) > 0
        and singular.min() > 0.85
        and singular.max() < 1.15
        and np.abs(warp[:, 2]).max() < 38
    )
    full_warp = warp.copy()
    full_warp[:, 2] *= width / 256
    validity = cv2.warpAffine(
        np.ones((height, width), np.uint8),
        full_warp,
        (width, height),
        flags=cv2.INTER_NEAREST | cv2.WARP_INVERSE_MAP,
    )
    overlap = float(validity.mean())
    aligned = cv2.warpAffine(
        ihc,
        full_warp,
        (width, height),
        flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
        borderValue=(255, 255, 255),
    )
    warped_proxy = proxy(aligned)
    support = cv2.resize(validity, (256, 256), interpolation=cv2.INTER_NEAREST) > 0
    after = float(np.corrcoef(template[support], warped_proxy[support])[0, 1])
    accepted = bool(
        safe
        and overlap > 0.85
        and np.isfinite(after)
        and after > max(0.3, before + 0.03)
    )
    report.update(
        accepted=accepted,
        overlap=overlap,
        after_correlation=after if np.isfinite(after) else None,
        ecc=float(correlation),
        warp=full_warp.tolist(),
    )
    if not accepted:
        return ihc, report
    # Return only an all-valid rectangle to the dataset cropper; no padded supervision.
    report["valid_rectangle"] = largest_valid_rectangle(validity)
    return aligned, report


def largest_valid_rectangle(mask):
    """Largest all-valid axis-aligned rectangle, returned as y0,x0,y1,x1."""
    scale = max(1, min(mask.shape) // 64)
    hh, ww = mask.shape[0] // scale, mask.shape[1] // scale
    mask = mask[: hh * scale, : ww * scale].reshape(hh, scale, ww, scale).min((1, 3))
    heights = np.zeros(mask.shape[1], dtype=int)
    best, rectangle = 0, (0, 0, 0, 0)
    for y, row in enumerate(mask):
        heights = np.where(row > 0, heights + 1, 0)
        stack = []
        for x, h in enumerate(np.append(heights, 0)):
            left = x
            while stack and stack[-1][1] > h:
                start, old = stack.pop()
                area = old * (x - start)
                left = start
                if area > best:
                    best = area
                    rectangle = (y - old + 1, start, y + 1, x)
            if not stack or stack[-1][1] < h:
                stack.append((left, int(h)))
    return [int(v) * scale for v in rectangle]


class RegisteredPairs(PairedDataset):
    """Lazy full-resolution training registration, cached before paired random crops.

    Crops stay within the valid overlap. Registration is never performed on val/test.
    Cache is keyed by manifest fingerprint in the caller's run directory.
    """

    def __init__(self, *args, cache, **kwargs):
        super().__init__(*args, **kwargs)
        if any(r["split"] != "train" for r in self.records):
            raise ValueError("Registration is restricted to training pairs")
        self.cache = Path(cache)
        self.cache.mkdir(parents=True, exist_ok=True)

    def __getitem__(self, index):
        number, seed = index if isinstance(index, tuple) else (index, 0)
        record = self.records[number]
        with Image.open(self.root / record["he"]) as image:
            he = np.array(image.convert("RGB"))
        cache = self.cache / (record["id"].replace("/", "__") + ".npz")
        report_path = cache.with_suffix(".json")
        if cache.exists() and report_path.exists():
            with np.load(cache) as stored:
                ihc = stored["ihc"]
            report = json.loads(report_path.read_text())
        else:
            with Image.open(self.root / record["ihc"]) as image:
                original = np.array(image.convert("RGB"))
            ihc, report = align_pair(he, original)
            rectangle = report.get("valid_rectangle", [0, 0, *he.shape[:2]])
            if min(rectangle[2] - rectangle[0], rectangle[3] - rectangle[1]) < (
                self.crop_size or he.shape[0]
            ):
                ihc = original
                report["accepted"] = False
                report.pop("valid_rectangle", None)
            np.savez_compressed(cache, ihc=ihc)
            report_path.write_text(json.dumps(report))
        generator = random.Random(seed)
        y0, x0, y1, x1 = report.get("valid_rectangle", [0, 0, *he.shape[:2]])
        size = self.crop_size or min(y1 - y0, x1 - x0)
        top = (
            generator.randrange(y0, y1 - size + 1)
            if self.augment
            else (y0 + y1 - size) // 2
        )
        left = (
            generator.randrange(x0, x1 - size + 1)
            if self.augment
            else (x0 + x1 - size) // 2
        )
        images = [a[top : top + size, left : left + size] for a in (he, ihc)]
        if self.augment:
            fx, fy, k = (
                generator.random() < 0.5,
                generator.random() < 0.5,
                generator.randrange(4),
            )
            images = [
                np.rot90(a[:: (-1 if fy else 1), :: (-1 if fx else 1)], k)
                for a in images
            ]
        tensors = [
            torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float() / 127.5
            - 1
            for a in images
        ]
        return dict(
            he=tensors[0], ihc=tensors[1], id=record["id"], group=record["group"]
        )

In [ ]:
preview = PairedDataset(DATA_ROOT,MANIFEST,"train",None)
fig,axes=plt.subplots(2,3,figsize=(11,7))
for row,index in enumerate(np.linspace(0,len(preview)-1,2,dtype=int)):
    item=preview[int(index)]
    he_u8=(rgb(item["he"])*255).round().astype(np.uint8)
    ihc_u8=(rgb(item["ihc"])*255).round().astype(np.uint8)
    aligned,report=align_pair(he_u8,ihc_u8)
    for ax,image,title in zip(axes[row],(he_u8,ihc_u8,aligned),("H&E","Original IHC",f"Affine accepted={report['accepted']}")):
        ax.imshow(image);ax.set_title(title);ax.axis("off")
    print(item["id"],report)
fig.tight_layout();plt.show();plt.close(fig)

## Training and checkpoint selection

AdamW, learning rate 1e-4, warmup 500, effective batch 4 and EMA 0.995. Auxiliary losses ramp up over 500 updates. Checkpoints use a proxy based on DAB and H&E structure error on 12 validation crops; inspect the gallery as well.


In [ ]:
def validation_data(root, manifest, settings):
    """Fixed group-balanced validation selection; no test data or quality-based selection."""
    dataset = PairedDataset(root, manifest, "val", settings["crop"])
    rng, grouped = random.Random(settings["seed"]), {}
    for record in dataset.records:
        grouped.setdefault(record["group"], []).append(record)
    groups = sorted(grouped)
    rng.shuffle(groups)
    for records in grouped.values():
        rng.shuffle(records)
    selected = []
    while len(selected) < min(settings["validation_count"], len(dataset)):
        for group in groups:
            if grouped[group] and len(selected) < settings["validation_count"]:
                selected.append(grouped[group].pop())
    dataset.records = selected
    return dataset


def pixel_scores(prediction, target):
    a, b = rgb(prediction), rgb(target)
    mse = float(np.square(a - b).mean())
    return {
        "rgb_mse": mse,
        "ssim": float(structural_similarity(a, b, channel_axis=-1, data_range=1)),
        "psnr": float(-10 * np.log10(mse)) if mse > 0 else None,
    }


@torch.inference_mode()
def validate(model, backbone, dataset, device, settings, directory):
    model.eval()
    rows = []
    for index in range(len(dataset)):
        item = dataset[index]
        he = item["he"].unsqueeze(0).to(device)
        noise = initial_noise(he, [item["id"]], settings["seed"])
        prediction = generate(model, backbone, he, noise, settings["ode_steps"])[
            0
        ].cpu()
        rows.append(
            {
                "id": item["id"],
                **pixel_scores(prediction, item["ihc"]),
                **proxy_scores(prediction, item["he"], item["ihc"]),
            }
        )
        if index < 4:
            save_image(
                torch.cat((item["he"], prediction, item["ihc"]), 2),
                directory / f"example-{index}.png",
            )
    write_json(directory / "metrics.json", rows)
    return {
        "val_ssim": float(np.mean([r["ssim"] for r in rows])),
        "val_rgb_mse": float(np.mean([r["rgb_mse"] for r in rows])),
        "val_dab_mae": float(np.mean([r["dab_mean_error"] for r in rows])),
        "val_he_structure": float(np.mean([r["he_structure_error"] for r in rows])),
    }


@torch.no_grad()
def optimizer_update(model, optimizer, scaler):
    """Let GradScaler skip an overflow before advancing EMA or the step counter."""
    scaler.unscale_(optimizer)
    parameters = [p for p in model.parameters() if p.grad is not None]
    finite = (
        torch.stack([torch.isfinite(p.grad).all() for p in parameters]).all().item()
    )
    if not finite:
        if not scaler.is_enabled():
            raise FloatingPointError("Non-finite gradients with AMP disabled")
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        return False
    # Finite gradients can still overflow a FP32 norm reduction.
    norm = torch.linalg.vector_norm(
        torch.stack(
            [torch.linalg.vector_norm(p.grad.to(torch.float64)) for p in parameters]
        )
    )
    coefficient = (1 / (norm + 1e-6)).clamp(max=1)
    for parameter in parameters:
        parameter.grad.mul_(coefficient.to(parameter.grad))
    scaler.step(optimizer)
    scaler.update()
    return True


def numpy_rng_state():
    name, keys, position, has_gauss, cached = np.random.get_state()
    return name, keys.tolist(), position, has_gauss, cached


def restore_numpy_rng(state):
    name, keys, position, has_gauss, cached = state
    np.random.set_state(
        (name, np.asarray(keys, dtype=np.uint32), position, has_gauss, cached)
    )


def save_checkpoint(path, payload):
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def train_session(
    root,
    manifest,
    directory,
    device,
    session_steps,
    micro_batch=1,
    resume=None,
    smoke=False,
    progress=True,
    experiment="full",
    backbone=None,
    backbone_path=None,
):
    """FM plus differentiable rollout supervision. Frozen backbone is external to checkpoints."""
    if session_steps < 0 or micro_batch < 1:
        raise ValueError("Invalid session step or micro-batch count")
    if session_steps == 0 and resume is None:
        raise ValueError("Inference-only mode requires a V4 checkpoint")
    settings = recipe(smoke, experiment)
    seed_everything(settings["seed"])
    backbone = (
        load_backbone(device, smoke, backbone_path) if backbone is None else backbone
    )
    backbone.eval().requires_grad_(False)
    model = UNet(settings["widths"], backbone.channels, experiment != "cnn-only").to(
        device
    )
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=settings["lr"], weight_decay=0.01
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    matcher = ConditionalFlowMatcher(sigma=0)
    fingerprint = sha256_file(manifest)
    step, batches_seen, best, skipped = 0, 0, -float("inf"), 0
    if resume is None and directory.exists() and any(directory.iterdir()):
        raise FileExistsError(
            "Non-empty run directory; specify its latest.pt or use a new directory"
        )
    if resume is not None:
        checkpoint = torch.load(resume, map_location="cpu", weights_only=True)
        if (
            checkpoint.get("recipe") != settings
            or checkpoint["manifest_sha256"] != fingerprint
        ):
            raise ValueError(
                "Resume requires the same V4 recipe and manifest; V1/V2/V3 are incompatible"
            )
        if checkpoint.get("backbone_sha256") != backbone.identity:
            raise ValueError("The frozen backbone weights differ from this checkpoint")
        model.load_state_dict(checkpoint["model"])
        ema.load_state_dict(checkpoint["ema"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        step, batches_seen = checkpoint["step"], checkpoint["batches_seen"]
        best, skipped = checkpoint["best_proxy_score"], checkpoint["amp_skipped"]
        torch.set_rng_state(checkpoint["torch_rng"])
        restore_numpy_rng(checkpoint["numpy_rng"])
        if device.type == "cuda" and checkpoint["cuda_rng"] is not None:
            torch.cuda.set_rng_state_all(
                checkpoint["cuda_rng"][: torch.cuda.device_count()]
            )
    directory.mkdir(parents=True, exist_ok=True)
    dataset_type = PairedDataset
    dataset_options = {}
    if experiment == "full-affine":
        dataset_type = RegisteredPairs
        dataset_options["cache"] = directory / "registration-cache" / fingerprint
    train_data = dataset_type(
        root,
        manifest,
        "train",
        settings["crop"],
        True,
        settings["limit_train"],
        **dataset_options,
    )
    val_data = validation_data(root, manifest, settings)
    if {r["group"] for r in train_data.records} & {
        r["group"] for r in val_data.records
    }:
        raise ValueError("Train and validation groups overlap")
    loader = DataLoader(
        train_data,
        batch_sampler=TrainingBatches(
            len(train_data), settings["batch"], settings["seed"], batches_seen
        ),
        num_workers=0,
        generator=torch.Generator().manual_seed(settings["seed"] + 1),
    )
    iterator = iter(loader)
    stop = min(settings["total_steps"], step + session_steps)

    def payload():
        return {
            "format": "structure_cfm_v4",
            "recipe": settings,
            "manifest_sha256": fingerprint,
            "model": model.state_dict(),
            "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "step": step,
            "batches_seen": batches_seen,
            "best_proxy_score": best,
            "amp_skipped": skipped,
            "torch_rng": torch.get_rng_state(),
            "backbone_sha256": backbone.identity,
            "numpy_rng": numpy_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all()
            if device.type == "cuda"
            else None,
        }

    write_json(directory / "recipe.json", settings)
    bar = tqdm(
        total=stop, initial=step, desc="Structure-CFM updates", disable=not progress
    )
    consecutive_skips = 0
    while step < stop:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        warmup = settings["warmup"]
        progress_fraction = (step - warmup) / max(1, settings["total_steps"] - warmup)
        factor = (
            (step + 1) / warmup
            if step < warmup
            else 0.5 * (1 + math.cos(math.pi * min(1, progress_fraction)))
        )
        for group in optimizer.param_groups:
            group["lr"] = settings["lr"] * factor
        batch = next(iterator)
        batches_seen += 1
        time, image, velocity, he = flow_batch(matcher, batch["he"], batch["ihc"])
        loss_sum = 0.0
        components = {}
        for first in range(0, len(image), micro_batch):
            section = slice(first, first + micro_batch)
            x, t, c, target = (
                a[section].to(device) for a in (image, time, he, velocity)
            )
            with torch.autocast(
                device.type, dtype=torch.float16, enabled=scaler.is_enabled()
            ):
                with torch.no_grad():
                    he_tokens = backbone(c)
                features = model.condition(c, he_tokens)
                prediction = model(x, t, c, features)
                fm_loss = F.mse_loss(prediction.float(), target.float())
            weight = len(x) / len(image)
            scaler.scale(fm_loss * weight).backward()
            auxiliary = {}
            # The rollout has its own graph: checkpoint recomputation bounds memory.
            if experiment != "fm-only":
                reference = batch["ihc"][section].to(device)
                with torch.autocast(
                    device.type, dtype=torch.float16, enabled=scaler.is_enabled()
                ):
                    features = model.condition(c, he_tokens)
                    endpoint = rollout(
                        model,
                        c,
                        torch.randn_like(c),
                        features,
                        settings["rollout_steps"],
                    )
                auxiliary = endpoint_losses(
                    endpoint.float(),
                    c,
                    reference,
                    backbone,
                    he_tokens,
                    semantic=experiment != "structure",
                )
                ramp = min(1.0, (step + 1) / settings["aux_ramp"])
                aux_loss = sum(
                    settings["loss_weights"][name] * value
                    for name, value in auxiliary.items()
                )
                if not torch.isfinite(aux_loss):
                    save_checkpoint(directory / "last-safe.pt", payload())
                    raise FloatingPointError("Non-finite endpoint loss")
                scaler.scale(aux_loss * ramp * weight).backward()
            loss = fm_loss.detach() + sum(
                settings["loss_weights"][name]
                * value.detach()
                * min(1.0, (step + 1) / settings["aux_ramp"])
                for name, value in auxiliary.items()
            )
            for name, value in {"fm": fm_loss, **auxiliary}.items():
                components[name] = (
                    components.get(name, 0.0) + value.detach().item() * weight
                )
            if not torch.isfinite(loss):
                save_checkpoint(directory / "last-safe.pt", payload())
                raise FloatingPointError("Non-finite FM loss")
            weight = len(x) / len(image)
            loss_sum += loss.detach().item() * weight
        try:
            updated = optimizer_update(model, optimizer, scaler)
        except FloatingPointError:
            save_checkpoint(directory / "last-safe.pt", payload())
            raise
        if not updated:
            skipped += 1
            consecutive_skips += 1
            print(
                f"AMP overflow: update skipped; scale={scaler.get_scale()}", flush=True
            )
            if consecutive_skips >= 16:
                save_checkpoint(directory / "last-safe.pt", payload())
                raise FloatingPointError("16 consecutive AMP overflows")
            continue
        consecutive_skips = 0
        with torch.no_grad():
            for average, current in zip(
                ema.parameters(), model.parameters(), strict=True
            ):
                average.lerp_(current, 1 - settings["ema"])
            for average, current in zip(ema.buffers(), model.buffers(), strict=True):
                average.copy_(current)
        step += 1
        record = {
            "step": step,
            "total_loss": loss_sum,
            **{f"loss_{name}": value for name, value in components.items()},
            "lr": optimizer.param_groups[0]["lr"],
            "amp_skipped": skipped,
        }
        if step % settings["validate_every"] == 0 or step == stop:
            record.update(
                validate(
                    ema,
                    backbone,
                    val_data,
                    device,
                    settings,
                    directory / "validation" / str(step),
                )
            )
            # Fixed proxy selection score; SSIM alone rewards matching misaligned geometry.
            score = -(record["val_dab_mae"] + record["val_he_structure"])
            if score > best:
                best = score
                save_checkpoint(directory / "best.pt", payload())
        if step % 200 == 0 or step == stop:
            save_checkpoint(directory / "latest.pt", payload())
        with (directory / "history.jsonl").open("a") as stream:
            stream.write(json.dumps(record) + "\n")
        bar.update(1)
        bar.set_postfix(loss=f"{loss_sum:.4f}", refresh=False)
        if step % 100 == 0 or step == stop:
            print(json.dumps(record), flush=True)
    bar.close()
    if resume is not None and stop <= checkpoint["step"]:
        return Path(resume)
    return directory / "latest.pt"

In [ ]:
resume=Path(RESUME_FROM) if RESUME_FROM else None
if resume is None and (RUN/"latest.pt").is_file():resume=RUN/"latest.pt"
if resume is not None and resume.parent.resolve()!=RUN.resolve():
    RUN.mkdir(parents=True,exist_ok=True)
    for name in ("history.jsonl","best.pt"):
        if (resume.parent/name).is_file() and not (RUN/name).exists():
            shutil.copy2(resume.parent/name,RUN/name)
checkpoint=train_session(DATA_ROOT,MANIFEST,RUN,DEVICE,SESSION_STEPS,
                         micro_batch=MICRO_BATCH,resume=resume,smoke=SMOKE_TEST,
                         experiment=EXPERIMENT,backbone=BACKBONE)
print("Checkpoint:",checkpoint)

## Validation results

Inspect raw and EMA outputs with the same noise. Look for missed strong staining, extra brown regions and lost structure. These previews are not a full-test benchmark.


In [ ]:
history=[json.loads(s) for s in (RUN/"history.jsonl").read_text().splitlines()] if (RUN/"history.jsonl").exists() else []
fig,axes=plt.subplots(1,3,figsize=(14,3.5))
for key in ("fm","content","structure","semantic","dab"):
    records=[r for r in history if f"loss_{key}" in r]
    axes[0].plot([r["step"] for r in records],[r[f"loss_{key}"] for r in records],label=key)
axes[0].legend();axes[0].set_title("Unweighted loss components")
records=[r for r in history if "val_ssim" in r]
for key in ("val_dab_mae","val_he_structure"):
    axes[1].plot([r["step"] for r in records],[r[key] for r in records],"o-",label=key)
axes[1].legend();axes[1].set_title("Validation proxies ↓")
axes[2].plot([r["step"] for r in records],[r["val_ssim"] for r in records],"o-");axes[2].set_title("SSIM: misaligned references")
fig.tight_layout();RUN.mkdir(parents=True,exist_ok=True);fig.savefig(RUN/"curves.png",dpi=140);plt.show();plt.close(fig)

state=torch.load(checkpoint,map_location="cpu",weights_only=True)
validation=validation_data(DATA_ROOT,MANIFEST,SETTINGS)
OUTPUT=WORK_ROOT/"v4-preview"/EXPERIMENT/f"step-{state['step']}"
OUTPUT.mkdir(parents=True,exist_ok=True)
# One network, two state dicts: avoids retaining two generators in GPU memory.
model=UNet(SETTINGS["widths"],BACKBONE.channels,EXPERIMENT!="cnn-only").to(DEVICE).eval().requires_grad_(False)
predictions={}
rows=[]
for weights in ("model","ema"):
    model.load_state_dict(state[weights])
    predictions[weights]=[]
    for index in range(len(validation)):
        item=validation[index];c=item["he"].unsqueeze(0).to(DEVICE)
        noise=initial_noise(c,[item["id"]],SETTINGS["seed"])
        prediction=generate(model,BACKBONE,c,noise,SETTINGS["ode_steps"])[0].cpu()
        predictions[weights].append(prediction)
        rows.append(dict(id=item["id"],group=item["group"],weights=weights,
                         **pixel_scores(prediction,item["ihc"]),**proxy_scores(prediction,item["he"],item["ihc"])))
        save_image(prediction,OUTPUT/"predictions"/weights/f"{item['id']}.png")
for first in range(0,len(validation),3):
    count=min(3,len(validation)-first);fig,axes=plt.subplots(count,4,figsize=(13,3.5*count),squeeze=False)
    for row,index in enumerate(range(first,first+count)):
        item=validation[index]
        for ax,image,title in zip(axes[row],(item["he"],item["ihc"],predictions["model"][index],predictions["ema"][index]),("H&E","Real IHC","Raw","EMA")):
            ax.imshow(rgb(image));ax.set_title(title);ax.axis("off")
        axes[row,0].text(0,-.07,item["id"],transform=axes[row,0].transAxes,fontsize=8)
    fig.tight_layout();fig.savefig(OUTPUT/f"gallery-{first//3+1:02d}.png",dpi=140);plt.show();plt.close(fig)
write_json(OUTPUT/"metrics.json",rows)

## Condition and seed swaps

Changing H&E should change tissue geometry in the output. A large RGB difference alone only shows sensitivity. Changing the seed at fixed H&E tests variation.


In [ ]:
model.load_state_dict(state["ema"])
item=validation[0]
other=next(validation[i] for i in range(1,len(validation)) if validation[i]["group"]!=item["group"])
c=item["he"].unsqueeze(0).to(DEVICE);changed=other["he"].unsqueeze(0).to(DEVICE)
noise=initial_noise(c,[item["id"]],42)
endpoint,frames=generate(model,BACKBONE,c,noise,SETTINGS["ode_steps"],trajectory=True)
fig,axes=plt.subplots(1,len(frames)+2,figsize=(3*(len(frames)+2),3))
panels=[(item["he"],"H&E")]+[(frame[0],f"ODE t={time:g}") for time,frame in frames]+[(item["ihc"],"Real IHC")]
for ax,(image,title) in zip(axes,panels):ax.imshow(rgb(image));ax.set_title(title);ax.axis("off")
fig.tight_layout();fig.savefig(OUTPUT/"trajectory.png",dpi=140);plt.show();plt.close(fig)
swapped=generate(model,BACKBONE,changed,noise,SETTINGS["ode_steps"])
reseeded=generate(model,BACKBONE,c,initial_noise(c,[item["id"]],43),SETTINGS["ode_steps"])
fig,axes=plt.subplots(1,5,figsize=(16,3.5))
for ax,image,title in zip(axes,(item["he"],other["he"],endpoint[0],swapped[0],reseeded[0]),("Original H&E","Different H&E","Original result","Changed H&E; same noise","Same H&E; different noise")):
    ax.imshow(rgb(image));ax.set_title(title);ax.axis("off")
fig.tight_layout();fig.savefig(OUTPUT/"condition-seed-swap.png",dpi=140);plt.show();plt.close(fig)
with torch.inference_mode():
    original_content=patch_content_loss(BACKBONE(endpoint),BACKBONE(c)).item()
    swapped_to_original=patch_content_loss(BACKBONE(swapped),BACKBONE(c)).item()
    swapped_to_changed=patch_content_loss(BACKBONE(swapped),BACKBONE(changed)).item()
write_json(OUTPUT/"condition-swap.json",dict(original_id=item["id"],changed_id=other["id"],
    input_rgb_mae=float(np.abs(rgb(item["he"])-rgb(other["he"])).mean()),
    output_rgb_mae=float(np.abs(rgb(endpoint[0])-rgb(swapped[0])).mean()),
    seed_rgb_mae=float(np.abs(rgb(endpoint[0])-rgb(reseeded[0])).mean()),
    original_content_error=original_content,swapped_to_original_error=swapped_to_original,
    swapped_to_changed_error=swapped_to_changed))

## DAB maps and solver study

Inspect DAB maps and compare Euler/Heun settings on three fixed validation pairs with shared seeds. Timings include warmup and GPU synchronization; this small study is exploratory.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(11,3.5))
for ax,image,title in zip(axes,(item["ihc"],endpoint[0].cpu(),swapped[0].cpu()),("Real DAB","Generated DAB","Swapped condition DAB")):
    dab=stain_channels(image.unsqueeze(0))[0,2].numpy()
    im=ax.imshow(dab,vmin=0,vmax=.3,cmap="magma");ax.set_title(title);ax.axis("off")
fig.colorbar(im,ax=axes.ravel().tolist(),shrink=.8);fig.savefig(OUTPUT/"dab.png",dpi=140);plt.show();plt.close(fig)

import time
variants=[("euler",SETTINGS["rollout_steps"]),("heun",8),("heun",32)] if not SMOKE_TEST else [("euler",2),("heun",4)]
benchmark=[]
for solver,steps in variants:
    generate(model,BACKBONE,c,noise,steps,solver=solver)
    for index in range(min(3,len(validation))):
        sample=validation[index];condition=sample["he"].unsqueeze(0).to(DEVICE)
        start_noise=initial_noise(condition,[sample["id"]],42)
        if DEVICE.type=="cuda":torch.cuda.synchronize()
        started=time.perf_counter()
        result=generate(model,BACKBONE,condition,start_noise,steps,solver=solver)[0]
        if DEVICE.type=="cuda":torch.cuda.synchronize()
        seconds=time.perf_counter()-started
        benchmark.append(dict(id=sample["id"],solver=solver,steps=steps,nfe=steps*(2 if solver=="heun" else 1),seconds=seconds,
                              **pixel_scores(result,sample["ihc"]),**proxy_scores(result.cpu(),sample["he"],sample["ihc"])))
write_json(OUTPUT/"ode-ablation.json",benchmark)
display(benchmark)

## Full-test export

After selecting a recipe on validation, export the unchanged test split. Use identical image processing for real and generated images. Published scores require matching splits and metric protocols. Cell-level HER2 localization has not been validated.


In [ ]:
@torch.inference_mode()
def export_split(split="test",weights="ema",solver="heun",steps=32):
    if split not in {"val","test"} or weights not in {"ema","model"}:raise ValueError("Invalid export")
    dataset=PairedDataset(DATA_ROOT,MANIFEST,split,SETTINGS["crop"],limit=4 if SMOKE_TEST else None)
    model.load_state_dict(state[weights]);destination=OUTPUT/f"export-{split}-{weights}-{solver}{steps}"
    records=[]
    for index in tqdm(range(len(dataset)),desc=f"Export {split}"):
        sample=dataset[index];condition=sample["he"].unsqueeze(0).to(DEVICE)
        noise=initial_noise(condition,[sample["id"]],42)
        result=generate(model,BACKBONE,condition,noise,steps,solver=solver)[0].cpu()
        save_image(result,destination/"generated"/f"{sample['id']}.png")
        save_image(sample["ihc"],destination/"real"/f"{sample['id']}.png")
        records.append(dict(id=sample["id"],group=sample["group"],**pixel_scores(result,sample["ihc"]),
                            **proxy_scores(result,sample["he"],sample["ihc"])))
    write_json(destination/"per-image.json",records)
    write_json(destination/"protocol.json",dict(split=split,weights=weights,solver=solver,steps=steps,
        crop=SETTINGS["crop"],count=len(dataset),manifest_sha256=sha256_file(MANIFEST),
        checkpoint_sha256=sha256_file(checkpoint),backbone_sha256=BACKBONE.identity))
    write_json(destination/"generated"/"sampling.json",dict(split=split,crop_size=SETTINGS["crop"],
        manifest_sha256=sha256_file(MANIFEST),limit=4 if SMOKE_TEST else None,solver=solver,steps=steps))
    return destination

def benchmark_export(destination,split="test",protocol="standard",features=True):
    from ihc_flow.metrics import evaluate
    if features and installed("pytorch-fid") is None:
        subprocess.run([sys.executable,"-m","pip","install","--quiet","--no-deps","pytorch-fid==0.3.0"],check=True)
    return evaluate(DATA_ROOT,MANIFEST,Path(destination)/"generated",Path(destination)/"evaluation",
                    DEVICE,split=split,protocol=protocol,features=features,cache_dir=WORK_ROOT/"metric-cache")
# После выбора модели:
# exported = export_split()
# report = benchmark_export(exported)

## Save outputs

Save checkpoints, recipe, history and previews. Resume also requires the same Phikon weights. The export archive excludes dataset images and the registration cache.


In [ ]:
archive_path=WORK_ROOT/f"her2-flow-v4-{EXPERIMENT}-step-{state['step']}.zip"
with zipfile.ZipFile(archive_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.iterdir()):
        if path.is_file():archive.write(path,path.relative_to(WORK_ROOT))
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file():archive.write(path,path.relative_to(WORK_ROOT))
print("Checkpoints:",RUN,"Preview:",OUTPUT)
display(FileLink(str(archive_path)))